# Demo Notebook — Biomedical Image Classification with CNNs

This notebook demonstrates how to:
1. Load the trained models (Custom CNN and ResNet-18)
2. Run inference on the PathMNIST test set
3. Evaluate performance with metrics and visualizations
4. Visualize predictions on individual test images
5. Compare model performance

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms

from src.dataset import load_pathmnist, create_dataloaders, CLASS_NAMES, IMAGENET_MEAN, IMAGENET_STD
from src.models import get_model, count_parameters
from src.evaluate import evaluate_model, compute_metrics, plot_confusion_matrix, plot_roc_curves

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Data

In [ ]:
# Load the PathMNIST dataset
train_ds, val_ds, test_ds = load_pathmnist(data_dir='../data', image_size=224)
_, _, test_loader = create_dataloaders(train_ds, val_ds, test_ds, batch_size=32, num_workers=2)

print(f'Test set size: {len(test_ds):,} images')
print(f'Number of classes: {len(CLASS_NAMES)}')

## 2. Load Trained Models

In [ ]:
# Load Custom CNN
custom_cnn = get_model('custom_cnn', num_classes=9).to(device)
custom_cnn.load_state_dict(torch.load('../results/models/best_custom_cnn.pth', map_location=device, weights_only=True))
custom_cnn.eval()

total, trainable = count_parameters(custom_cnn)
print(f'Custom CNN — Total params: {total:,}, Trainable: {trainable:,}')

# Load ResNet-18
resnet18 = get_model('resnet18', num_classes=9).to(device)
resnet18.load_state_dict(torch.load('../results/models/best_resnet18.pth', map_location=device, weights_only=True))
resnet18.eval()

total, trainable = count_parameters(resnet18)
print(f'ResNet-18  — Total params: {total:,}, Trainable: {trainable:,}')

## 3. Evaluate Both Models on Test Set

In [ ]:
# Evaluate Custom CNN
cnn_labels, cnn_preds, cnn_probs = evaluate_model(custom_cnn, test_loader, device)
cnn_metrics = compute_metrics(cnn_labels, cnn_preds, cnn_probs)

print('Custom CNN Test Results:')
print(f'  Accuracy:  {cnn_metrics["accuracy"]:.4f}')
print(f'  Precision: {cnn_metrics["precision"]:.4f}')
print(f'  Recall:    {cnn_metrics["recall"]:.4f}')
print(f'  F1 Score:  {cnn_metrics["f1_score"]:.4f}')
print(f'  ROC AUC:   {cnn_metrics["roc_auc"]:.4f}')

In [ ]:
# Evaluate ResNet-18
res_labels, res_preds, res_probs = evaluate_model(resnet18, test_loader, device)
res_metrics = compute_metrics(res_labels, res_preds, res_probs)

print('ResNet-18 Test Results:')
print(f'  Accuracy:  {res_metrics["accuracy"]:.4f}')
print(f'  Precision: {res_metrics["precision"]:.4f}')
print(f'  Recall:    {res_metrics["recall"]:.4f}')
print(f'  F1 Score:  {res_metrics["f1_score"]:.4f}')
print(f'  ROC AUC:   {res_metrics["roc_auc"]:.4f}')

## 4. Model Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'],
    'Custom CNN': [
        cnn_metrics['accuracy'], cnn_metrics['precision'],
        cnn_metrics['recall'], cnn_metrics['f1_score'], cnn_metrics['roc_auc']
    ],
    'ResNet-18': [
        res_metrics['accuracy'], res_metrics['precision'],
        res_metrics['recall'], res_metrics['f1_score'], res_metrics['roc_auc']
    ],
})
comparison['Difference'] = comparison['ResNet-18'] - comparison['Custom CNN']
print(comparison.to_string(index=False, float_format='%.4f'))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison))
width = 0.35
ax.bar(x - width/2, comparison['Custom CNN'], width, label='Custom CNN', color='steelblue')
ax.bar(x + width/2, comparison['ResNet-18'], width, label='ResNet-18', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Metric'])
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

for ax, (name, labels, preds) in zip(axes, [
    ('Custom CNN', cnn_labels, cnn_preds),
    ('ResNet-18', res_labels, res_preds)
]):
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig('../results/comparison_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sample Predictions on Test Images

In [ ]:
def denormalize(tensor):
    """Reverse ImageNet normalization for visualization."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

# Select 8 random test samples
np.random.seed(42)
indices = np.random.choice(len(test_ds), 8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, ax in zip(indices, axes.flat):
    img, label = test_ds[idx]
    label = label.item() if hasattr(label, 'item') else int(label)
    
    # Get predictions from both models
    with torch.no_grad():
        img_input = img.unsqueeze(0).to(device)
        cnn_pred = custom_cnn(img_input).argmax(dim=1).item()
        res_pred = resnet18(img_input).argmax(dim=1).item()
    
    # Display image
    img_display = denormalize(img).permute(1, 2, 0).numpy()
    ax.imshow(img_display)
    
    color = 'green' if (cnn_pred == label and res_pred == label) else 'red'
    ax.set_title(
        f'True: {CLASS_NAMES[label]}\n'
        f'CNN: {CLASS_NAMES[cnn_pred]}\n'
        f'ResNet: {CLASS_NAMES[res_pred]}',
        fontsize=8, color=color
    )
    ax.axis('off')

plt.suptitle('Sample Test Predictions', fontsize=14)
plt.tight_layout()
plt.savefig('../results/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Per-Class Performance Analysis

In [ ]:
from sklearn.metrics import classification_report

print('=== Custom CNN — Per-Class Report ===')
print(classification_report(cnn_labels, cnn_preds, target_names=CLASS_NAMES, zero_division=0))

print('\n=== ResNet-18 — Per-Class Report ===')
print(classification_report(res_labels, res_preds, target_names=CLASS_NAMES, zero_division=0))

## 8. ROC Curves Comparison

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (name, labels, probs) in zip(axes, [
    ('Custom CNN', cnn_labels, cnn_probs),
    ('ResNet-18', res_labels, res_probs)
]):
    labels_bin = label_binarize(labels, classes=list(range(9)))
    for i in range(9):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], probs[:, i])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{CLASS_NAMES[i]} ({roc_auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{name} — ROC Curves')
    ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig('../results/comparison_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

This demo notebook demonstrates the full inference and evaluation pipeline for both the Custom CNN (trained from scratch) and the pretrained ResNet-18 (fine-tuned) on the PathMNIST dataset.

**Key Takeaways:**
- Transfer learning with ResNet-18 is expected to outperform the custom CNN, especially given limited training data.
- Both models can be loaded from saved checkpoints and used for inference on new pathology images.
- The confusion matrices and ROC curves help identify which tissue classes are most challenging to distinguish.